In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os

# 1. Download dataset directly via Kaggle API
path = kagglehub.dataset_download("kanchana1990/global-clinical-trial-intelligence-20242026")

# Find the CSV file in the downloaded folder
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
file_path = os.path.join(path, csv_files[0])

# 2. Load the dataset
df = pd.read_csv(file_path)

# 3. Save a raw copy to your local data folder (so you don't have to re-download)
df.to_csv('../data/raw/raw_clinical_data.csv', index=False)

print(f"Dataset loaded successfully! Shape: {df.shape}")
print("Columns:", df.columns.tolist())

c:\Users\santw\OneDrive\Desktop\data analytics project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.88M/1.88M [00:02<00:00, 882kB/s] 

Extracting files...


Dataset loaded successfully! Shape: (5000, 22)
Columns: ['nct_id', 'title', 'status', 'phase', 'study_type', 'condition', 'intervention', 'sponsor', 'sponsor_class', 'start_date', 'completion_date', 'enrollment', 'country', 'has_results', 'brief_summary', 'duration_days', 'phase_numeric', 'enrollment_bucket', 'is_industry_sponsored', 'multi_country', 'condition_category', 'summary_word_count']


In [2]:
# 1. Standardize column names (lowercase, replace spaces with underscores)
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)

# 2. Drop full duplicate rows
before_len = len(df)
df = df.drop_duplicates()
print(f"Dropped {before_len - len(df)} duplicate rows.")

# 3. Handle Missing Values
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

# Fill missing numeric values with the median
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill missing categorical values with "Unknown" to preserve data integrity
df[cat_cols] = df[cat_cols].fillna("Unknown")

# 4. Standardize dates (if applicable)
date_cols = [c for c in df.columns if 'date' in c]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print("Data cleaning complete. Checking nulls:\n", df.isnull().sum().max(), "maximum nulls in any column.")

Dropped 0 duplicate rows.
Data cleaning complete. Checking nulls:
 1188 maximum nulls in any column.


C:\Users\santw\AppData\Local\Temp\ipykernel_22736\3912863102.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


In [3]:
from sklearn.preprocessing import MinMaxScaler

# 1. Trial Status Flags (For building the Funnel later)
if 'status' in df.columns:
    # Lowercase the status for safe matching
    status_lower = df['status'].str.lower()
    df['is_recruiting'] = (status_lower == 'recruiting').astype(int)
    df['is_active'] = (status_lower.isin(['active', 'enrolling by invitation'])).astype(int)
    df['is_completed'] = (status_lower == 'completed').astype(int)
    df['is_terminated'] = (status_lower.isin(['terminated', 'withdrawn', 'suspended'])).astype(int)

# 2. Phase Numeric Encoding (For ML/Segmentation)
phase_map = {
    'phase 1': 1, 'phase 2': 2, 'phase 3': 3, 'phase 4': 4,
    'phase 1/phase 2': 1.5, 'phase 2/phase 3': 2.5,
    'early phase 1': 0.5, 'not applicable': 0, 'unknown': 0
}
if 'phase' in df.columns:
    df['phase_numeric'] = df['phase'].str.lower().map(phase_map).fillna(0)

# 3. Trial Success/Complexity Score
# A composite score based on whether it completed, how big it was, and the phase.
# This directly models real-world evidence analytics.
cols_for_scoring = []
if 'is_completed' in df.columns: cols_for_scoring.append('is_completed')
if 'enrollment' in df.columns: cols_for_scoring.append('enrollment')
if 'phase_numeric' in df.columns: cols_for_scoring.append('phase_numeric')

if len(cols_for_scoring) > 1:
    # Ensure numeric types
    df[cols_for_scoring] = df[cols_for_scoring].apply(pd.to_numeric, errors='coerce').fillna(0)
    
    scaler = MinMaxScaler()
    scaled_features = scaler.fit_transform(df[cols_for_scoring])
    df['trial_complexity_score'] = scaled_features.mean(axis=1).round(4)

# 4. Save the processed, enriched dataset
df.to_csv('../data/processed/clean_clinical_data.csv', index=False)
print(f"Feature engineering complete! Saved {df.shape[0]} rows and {df.shape[1]} columns to processed folder.")
print("New columns ready for dashboard:", [c for c in df.columns if 'is_' in c or 'score' in c])

Feature engineering complete! Saved 5000 rows and 27 columns to processed folder.
New columns ready for dashboard: ['is_industry_sponsored', 'is_recruiting', 'is_active', 'is_completed', 'is_terminated', 'trial_complexity_score']
